# volas quickstart

Run a small OHLCV indicator pipeline in the browser: build a frame, compute cached indicators, append one new bar, refresh only the stale tail, and export a NumPy feature matrix.

## Install

Colab already ships with NumPy loaded by parts of the runtime, so install only the published volas wheel. Upgrading NumPy inside the notebook can force a runtime restart.

In [ ]:
import sys

assert sys.version_info >= (3, 11), "volas requires Python 3.11 or newer"
%pip install -U volas

## Build a synthetic OHLCV frame

The demo is self-contained: no data file, token, exchange account, or network data source is required after installation.

In [ ]:
import numpy as np
from volas import DataFrame

n = 80
t = np.arange(n, dtype=float)
close = 100.0 + np.cumsum(0.15 + 0.75 * np.sin(t / 6.0))

df = DataFrame({
    "open": close - 0.25,
    "high": close + 0.65,
    "low": close - 0.70,
    "close": close,
    "volume": 1_000.0 + 5.0 * t,
})

df.tail(3)

## Compute indicator columns

A directive such as `rsi:14` or `macd.signal` behaves like a column request. volas computes it once and keeps the result cached on the frame.

In [ ]:
indicators = df[[
    "rsi:14",
    "macd",
    "macd.signal",
    "macd.histogram",
    "atr:14",
    "boll",
    "boll.upper",
    "boll.lower",
]]

indicators.tail(5)

## Append one new bar

After `append`, cached indicator columns are stale only at the affected tail. Reading the indicator columns refreshes that tail instead of recomputing the full history.

In [ ]:
new_close = float(close[-1] + 0.45)
new_bar = DataFrame({
    "open": [float(close[-1])],
    "high": [new_close + 0.50],
    "low": [new_close - 0.60],
    "close": [new_close],
    "volume": [1_700.0],
})

df.append(new_bar)

updated = df[["close", "rsi:14", "macd", "macd.signal", "macd.histogram", "atr:14"]]
updated.tail(5)

## Export model-ready features

`to_numpy()` returns a dense matrix that can feed NumPy or Torch pipelines.

In [ ]:
feature_matrix = updated.to_numpy()

assert len(df) == n + 1
assert feature_matrix.shape == (len(df), 6)

feature_matrix.shape